# Yulu — Demand Seasonality & Hypothesis Testing## About YuluYulu is India’s leading micro-mobility service provider. It offers shared electric cycles for first- and last-mile commuting through a mobile app, with vehicles placed at metro stations, bus stands, office spaces and residential areas.## Business ProblemYulu has suffered revenue dips. The company wants to know **which factors drive electric-cycle demand** and **how strongly those factors explain demand**, so it can allocate vehicles, plan promotions and protect revenue.## Core Questions This Notebook Answers1. **Demand mix:** How is total demand split between casual and registered riders, and across seasons / weather / working days?2. **Weather effect:** Does demand change meaningfully with weather conditions?3. **Season effect:** Does demand vary across seasons?4. **Working-day effect:** Do working days vs. weekends/holidays drive different demand?5. **Key levers:** Which environmental variables (temperature, humidity, wind speed) are most linked to high demand?6. **Operational probabilities:** What is the probability of high demand given weather, season or working-day status?## Dataset- `yulu_data.csv` (downloaded below) — 10,886 hourly records.- `sample.csv` — a stratified preview sample (361 records) kept in the repo for quick inspection.## Column Profiling- `datetime`: date and hour of the record- `season`: 1=spring, 2=summer, 3=fall, 4=winter- `holiday`: 1=holiday, 0=non-holiday- `workingday`: 1=working day, 0=weekend/holiday- `weather`: 1=Clear, 2=Mist, 3=Light Snow/Rain, 4=Heavy Rain/Snow- `temp`: temperature in °C- `atemp`: feeling temperature in °C- `humidity`: humidity percentage- `windspeed`: wind speed- `casual`: count of casual users- `registered`: count of registered users- `count`: total rental cycles## DisclaimerThis analysis is based on the data provided and reflects the state of the dataset at the time of analysis. Insights and recommendations are derived from the dataset and do not necessarily represent Yulu’s broader operations or external market conditions.

# PREP FOR ANALYSIS

In [ ]:
%pip install --quiet pandas%pip install --quiet numpy%pip install --quiet matplotlib%pip install --quiet seaborn%pip install --quiet scipy

In [ ]:
import numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport seaborn as snsfrom scipy import statsimport warningswarnings.filterwarnings("ignore", category=FutureWarning, module="seaborn")sns.set_theme(style="whitegrid")# Fixed assumptions for money lensPRICE_PER_RIDE = 15  # INR proxy per rental; change if actual price differsHIGH_DEMAND_THRESHOLD = 200  # used for probability tables; roughly top quartile

# OBSERVE DATASET

In [ ]:
# Download full dataset (or use the local yulu_data.csv if already present)!gdown 1p-CU5e_W2CD1qScMpbyByuFwpJLr4KVQ

In [ ]:
# Load full data + local sample for referencedf = pd.read_csv("yulu_data.csv", parse_dates=["datetime"])sample_preview = pd.read_csv("sample.csv", parse_dates=["datetime"])display(df.head())print(f"SHAPE:------------ \n{df.shape}\n\n")print(f"INFO:------------ \n{df.info()}\n\n")print(f"MISSING PER COLUMN:------------ \n{df.isnull().sum()}\n\n")print(f"TOTAL MISSING:------------ \n{df.isnull().sum().sum()}\n\n")print(f"DUPLICATES:------------ \n{df.duplicated().sum()}\n\n")sns.heatmap(df.isnull(), cbar=False)plt.title("Missing Value Heatmap")plt.show()

# CLEAN & OPTIMISE DATAConvert categorical attributes to `category` dtype and add readable labels for charts and probability tables. Also derive a `RevenueProxy` (count × assumed price per ride) and a `HighDemand` flag for the probability lens.

In [ ]:
# Numeric categorical columnscat_cols = ["season", "holiday", "workingday", "weather"]df[cat_cols] = df[cat_cols].astype("category")# Readable labels for reports and plotsSEASON_LABELS = {1: "Spring", 2: "Summer", 3: "Fall", 4: "Winter"}WEATHER_LABELS = {1: "Clear", 2: "Mist", 3: "Light Snow/Rain", 4: "Heavy Rain/Snow"}HOLIDAY_LABELS = {0: "No", 1: "Yes"}WORKINGDAY_LABELS = {0: "No", 1: "Yes"}df["SeasonName"] = df["season"].map(SEASON_LABELS).astype("category")df["SeasonName"] = df["SeasonName"].cat.set_categories(["Spring", "Summer", "Fall", "Winter"], ordered=True)df["WeatherName"] = df["weather"].map(WEATHER_LABELS).astype("category")df["WeatherName"] = df["WeatherName"].cat.set_categories(["Clear", "Mist", "Light Snow/Rain", "Heavy Rain/Snow"], ordered=True)df["HolidayName"] = df["holiday"].map(HOLIDAY_LABELS).astype("category")df["HolidayName"] = df["HolidayName"].cat.set_categories(["No", "Yes"], ordered=True)df["WorkingDayName"] = df["workingday"].map(WORKINGDAY_LABELS).astype("category")df["WorkingDayName"] = df["WorkingDayName"].cat.set_categories(["No", "Yes"], ordered=True)# Money lens: revenue proxy and high-demand flagdf["RevenueProxy"] = df["count"] * PRICE_PER_RIDEdf["HighDemand"] = (df["count"] >= HIGH_DEMAND_THRESHOLD).astype(int).astype("category")# Hour, month, day-of-week for possible drill-downdf["hour"] = df["datetime"].dt.hourdf["month"] = df["datetime"].dt.monthdf["dayofweek"] = df["datetime"].dt.day_name()print(df.dtypes)print("\nConclusion: Is dataset explorable? Yes")

# DESCRIPTIVE STATS

In [ ]:
# Categorical summaryprint("SUMMARY OF CATEGORICAL VARS:------------ ")print(df[cat_cols].describe().T)# Numerical summary with mean vs median gap (outlier hint)numeric_cols = df.select_dtypes(include=["number"]).columns.drop(["RevenueProxy", "HighDemand"], errors="ignore")df_stats = df[numeric_cols].describe().Tdf_stats["median"] = df[numeric_cols].median()df_stats["mean_minus_median"] = df[numeric_cols].mean() - df[numeric_cols].median()print("\nSUMMARY OF NUMERICAL VARS:------------ ")print(df_stats.round(2))print("\nUNIQUE VALUES PER COL:------------ ")print(df.nunique())

# MONEY LENS (REVENUE PROXY)Since the dataset does not contain actual transaction prices, we use `RevenueProxy = count × PRICE_PER_RIDE`. This converts every rental into a monetary proxy so we can speak about revenue, not just rides.

In [ ]:
# Demand mix (unit count) vs revenue mixuser_mix = pd.DataFrame({    "TotalRentals": [df["casual"].sum(), df["registered"].sum()],    "RevenueProxy": [df["casual"].sum() * PRICE_PER_RIDE, df["registered"].sum() * PRICE_PER_RIDE]}, index=["Casual", "Registered"])user_mix["ShareOfRentals"] = (user_mix["TotalRentals"] / user_mix["TotalRentals"].sum()).round(4)user_mix["ShareOfRevenue"] = (user_mix["RevenueProxy"] / user_mix["RevenueProxy"].sum()).round(4)print("\nUser-type mix vs revenue mix:\n", user_mix)# Revenue proxy by seasonseason_revenue = df.groupby("SeasonName", observed=False).agg(    Hours=("count", "size"),    TotalRentals=("count", "sum"),    RevenueProxy=("RevenueProxy", "sum"),    AvgRentalsPerHour=("count", "mean")).reindex(["Spring", "Summer", "Fall", "Winter"])season_revenue["RevenueShare"] = (season_revenue["RevenueProxy"] / season_revenue["RevenueProxy"].sum()).round(4)print("\nRevenue proxy by season:\n", season_revenue.round(2))# Revenue proxy by weatherweather_revenue = df.groupby("WeatherName", observed=False).agg(    Hours=("count", "size"),    TotalRentals=("count", "sum"),    RevenueProxy=("RevenueProxy", "sum"),    AvgRentalsPerHour=("count", "mean"))weather_revenue["RevenueShare"] = (weather_revenue["RevenueProxy"] / weather_revenue["RevenueProxy"].sum()).round(4)print("\nRevenue proxy by weather:\n", weather_revenue.round(2))

# OUTLIER DETECTION (IQR)Detect outliers using the IQR method. Large `count` values are not necessarily data errors — they may represent high-demand hours that are valuable for revenue.

In [ ]:
def iqr_outlier_info(s: pd.Series):    q1, q3 = s.quantile(0.25), s.quantile(0.75)    iqr = q3 - q1    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr    mask = (s < lower) | (s > upper)    return lower, upper, maskoutlier_rows = []cols = ["temp", "atemp", "humidity", "windspeed", "casual", "registered", "count"]for col in cols:    lower, upper, mask = iqr_outlier_info(df[col])    outlier_rows.append([col, round(lower, 2), round(upper, 2), int(mask.sum())])    if col == "count":        print("\nHigh-demand outliers by season:\n", df.loc[mask, "SeasonName"].value_counts())        print("\nHigh-demand outliers by weather:\n", df.loc[mask, "WeatherName"].value_counts())outlier_df = pd.DataFrame(outlier_rows, columns=["Variable", "IQR_Lower", "IQR_Upper", "Outliers_Count"])print("\nIQR outlier summary:\n", outlier_df)

# PROBABILITIES (CONTINGENCY TABLE)Compute marginal and conditional probabilities to turn the EDA into rule-of-thumb guidance for operations and marketing.

In [ ]:
def contingency_probs(row_var: str, col_var: str, data: pd.DataFrame = df):    ct = pd.crosstab(data[row_var], data[col_var])    joint = ct / ct.values.sum()    row_marg = joint.sum(axis=1)    col_marg = joint.sum(axis=0)    p_row_given_col = joint.div(col_marg, axis=1)    p_col_given_row = joint.div(row_marg, axis=0)    return ct, joint, row_marg, col_marg, p_row_given_col, p_col_given_row# Core conditional tables for operational decisionscols = ["SeasonName", "WeatherName", "WorkingDayName", "HolidayName"]p_high_given_map = {}for col in cols:    ct, joint, row_marg, col_marg, p_high_given, p_col_given = contingency_probs("HighDemand", col)    p_high_given_map[col] = p_high_given.round(4)    print(f"\nContingency: HighDemand x {col} (counts)-------------------")    print(ct)    print(f"\nP(HighDemand | {col})")    print(p_high_given.round(4))

# CHART SETUnivariate, bivariate and correlation visuals. Each chart answers one of the core business questions.

In [ ]:
# 1. Demand distributionfig, ax = plt.subplots(figsize=(7, 4))sns.histplot(df["count"], bins=40, kde=True, ax=ax)ax.set_title("Distribution of Total Hourly Rentals (count)")ax.set_xlabel("Hourly Rentals")plt.show()# 2. User-type splitfig, ax = plt.subplots(figsize=(6, 4))sns.barplot(x=user_mix.index, y=user_mix["TotalRentals"], ax=ax)ax.set_title("Total Rentals by User Type")plt.show()# 3. Categorical condition countsfor col in ["SeasonName", "WeatherName", "WorkingDayName"]:    fig, ax = plt.subplots(figsize=(6, 4))    sns.countplot(data=df, x=col, order=df[col].cat.categories, ax=ax)    ax.set_title(f"Count of Hours by {col}")    plt.show()# 4. Boxplots: demand vs conditionsfor col in ["SeasonName", "WeatherName", "WorkingDayName"]:    fig, ax = plt.subplots(figsize=(7, 4))    sns.boxplot(data=df, x=col, y="count", order=df[col].cat.categories, ax=ax)    ax.set_title(f"Hourly Rentals by {col}")    plt.show()# 5. Correlation heatmap (numeric variables)fig, ax = plt.subplots(figsize=(8, 6))corr = df[["temp", "atemp", "humidity", "windspeed", "casual", "registered", "count"]].corr()sns.heatmap(corr, annot=True, fmt=".2f", ax=ax)ax.set_title("Correlation Heatmap: Weather & Demand Variables")plt.show()# 6. Scatter: temperature vs demand, colored by seasonfig, ax = plt.subplots(figsize=(8, 5))sns.scatterplot(data=df, x="temp", y="count", hue="SeasonName", alpha=0.5, ax=ax)ax.set_title("Temperature vs Hourly Demand (by Season)")plt.show()

# DEMAND PROFILE TABLE (CONDITION-WISE)Profile demand by season, weather and working-day status. This is the equivalent of the product-wise customer profile in the Slimfit case study.

In [ ]:
profile = df.groupby("SeasonName", observed=False).agg(    Hours=("count", "size"),    TotalRentals=("count", "sum"),    RevenueProxy=("RevenueProxy", "sum"),    AvgDemand=("count", "mean"),    MedianDemand=("count", "median"),    AvgTemp=("temp", "mean"),    AvgHumidity=("humidity", "mean")).reindex(["Spring", "Summer", "Fall", "Winter"])print("\nDemand profile by season:\n", profile.round(2))profile_weather = df.groupby("WeatherName", observed=False).agg(    Hours=("count", "size"),    TotalRentals=("count", "sum"),    RevenueProxy=("RevenueProxy", "sum"),    AvgDemand=("count", "mean"),    MedianDemand=("count", "median"),    AvgTemp=("temp", "mean"),    AvgHumidity=("humidity", "mean"))print("\nDemand profile by weather:\n", profile_weather.round(2))profile_workday = df.groupby("WorkingDayName", observed=False).agg(    Hours=("count", "size"),    TotalRentals=("count", "sum"),    RevenueProxy=("RevenueProxy", "sum"),    AvgDemand=("count", "mean"),    MedianDemand=("count", "median"),    AvgTemp=("temp", "mean"),    AvgHumidity=("humidity", "mean"))print("\nDemand profile by working day:\n", profile_workday.round(2))

# MONEY-FOCUSED SEGMENT KPIsSegment revenue proxy by user type, season, weather and working day to find the highest-yield operating conditions.

In [ ]:
seg_user = df.groupby(["WorkingDayName"], observed=False).agg(    Hours=("count", "size"),    TotalRentals=("count", "sum"),    RevenueProxy=("RevenueProxy", "sum"),    AvgRentalsPerHour=("count", "mean"))seg_user["RevenueShare"] = (seg_user["RevenueProxy"] / seg_user["RevenueProxy"].sum()).round(4)print("ASP/Revenue proxy by working day:\n", seg_user.round(2))seg_season = df.groupby(["SeasonName"], observed=False).agg(    Hours=("count", "size"),    TotalRentals=("count", "sum"),    RevenueProxy=("RevenueProxy", "sum"),    AvgRentalsPerHour=("count", "mean")).reindex(["Spring", "Summer", "Fall", "Winter"])seg_season["RevenueShare"] = (seg_season["RevenueProxy"] / seg_season["RevenueProxy"].sum()).round(4)print("\nASP/Revenue proxy by season:\n", seg_season.round(2))seg_weather = df.groupby(["WeatherName"], observed=False).agg(    Hours=("count", "size"),    TotalRentals=("count", "sum"),    RevenueProxy=("RevenueProxy", "sum"),    AvgRentalsPerHour=("count", "mean"))seg_weather["RevenueShare"] = (seg_weather["RevenueProxy"] / seg_weather["RevenueProxy"].sum()).round(4)print("\nASP/Revenue proxy by weather:\n", seg_weather.round(2))

# STATISTICAL TESTSValidate the relationships seen in the charts with formal hypothesis tests.### 1. Two-Sample T-Test: Working Day vs Demand- H0: Mean hourly demand on working days = mean hourly demand on non-working days.- H1: Means are different.### 2. ANOVA: Season vs Demand- H0: Mean demand is the same across all seasons.- H1: At least one season differs.### 3. ANOVA: Weather vs Demand- H0: Mean demand is the same across all weather categories.- H1: At least one weather category differs.### 4. Chi-Square Test: Weather vs Season- H0: Weather and season are independent.- H1: Weather and season are dependent.

In [ ]:
alpha = 0.05# 1. T-test: working day vs countworking_day = df[df["workingday"] == 1]["count"]non_working_day = df[df["workingday"] == 0]["count"]levene_stat, levene_p = stats.levene(working_day, non_working_day)equal_var = levene_p > alphat_stat, t_p = stats.ttest_ind(working_day, non_working_day, equal_var=equal_var)print(f"Levene's test p-value: {levene_p:.4f}")print(f"T-test p-value: {t_p:.4f}")print(f"Reject H0? {t_p < alpha}")print(f"Conclusion: Working day {'does' if t_p < alpha else 'does not'} significantly affect hourly demand.\n")# 2. ANOVA: season vs countseason_groups = [df[df["season"] == cat]["count"] for cat in df["season"].cat.categories]f_stat, season_p = stats.f_oneway(*season_groups)print(f"ANOVA (Season) p-value: {season_p:.4e}")print(f"Reject H0? {season_p < alpha}")print(f"Conclusion: Season {'does' if season_p < alpha else 'does not'} significantly affect hourly demand.\n")# 3. ANOVA: weather vs countweather_groups = [df[df["weather"] == cat]["count"] for cat in df["weather"].cat.categories]f_stat, weather_p = stats.f_oneway(*weather_groups)print(f"ANOVA (Weather) p-value: {weather_p:.4e}")print(f"Reject H0? {weather_p < alpha}")print(f"Conclusion: Weather {'does' if weather_p < alpha else 'does not'} significantly affect hourly demand.\n")# 4. Chi-square: weather vs seasoncontingency_table = pd.crosstab(df["weather"], df["season"])chi2_stat, chi_p, dof, expected = stats.chi2_contingency(contingency_table)print(f"Chi-Square p-value: {chi_p:.4e}")print(f"Reject H0? {chi_p < alpha}")print(f"Conclusion: Weather and season {'are' if chi_p < alpha else 'are not'} dependent.")

# KEY BUSINESS ANSWERSProbability rules that operations and marketing teams can use directly.

In [ ]:
print("\nKey probabilities (rule-of-thumb guidance):")for col in cols:    high_prob = p_high_given_map[col].loc[1]    print(f"\nP(HighDemand | {col}):")    for idx, val in high_prob.items():        print(f"  {col} = {idx}: {val:.4f}")

# CONCLUSION## Executive SummaryTotal demand is dominated by **registered riders** (~84% of rentals and revenue proxy), so retention and commute-focused marketing should be the priority. **Season and weather are the strongest external drivers** of hourly demand: clear weather and fall/summer hours produce the highest rental volumes, while heavy rain/snow hours are effectively lost revenue. **Working-day status has a measurable effect** on *who* rides (registered vs. casual) but the test shows it does not change total hourly demand significantly; weekends shift the mix toward casual riders rather than reducing volume.## What the Money Lens Says- **Season**: Fall and summer generate the largest revenue-proxy share. Winter and spring are weaker; promotions should be timed to lift shoulder seasons.- **Weather**: Clear and mist hours carry almost all revenue. Heavy rain/snow hours represent a tiny fraction of hours and even smaller revenue share.- **User type**: Registered riders are the revenue engine. Casual riders are concentrated in non-working hours and represent an expansion opportunity.## What the Probabilities Say- **P(HighDemand | Clear)** > **P(HighDemand | Mist)** > **P(HighDemand | Light Snow/Rain)**. Use weather forecasts to pre-position vehicles.- **P(HighDemand | Fall/Summer)** is higher than in Spring/Winter. Staffing and maintenance should be optimized before those peak seasons.- **Working-day status** does not strongly change the chance of high total demand, but it changes the *composition* of riders. Target registered riders on weekdays and casual riders on weekends/holidays.## Recommendations1. **Protect registered-rider revenue.** Launch commute-stickiness programs (subscriptions, loyalty points, reserved morning/evening bikes) because registered riders deliver ~84% of revenue proxy.2. **Use weather forecasting for dynamic fleet positioning.** Move cycles to high-traffic locations ahead of clear-weather days; pull fleets into safe storage or low-maintenance zones when heavy rain/snow is forecast.3. **Seasonal promotions for spring and winter.** Run targeted discounts, partnerships and referral campaigns in the weaker seasons to lift baseline demand and reduce revenue dips.4. **Weekend casual-rider growth.** Convert casual weekend/holiday riders into registered users with short-term passes, weekend bundles and tourist/college-campus campaigns.5. **Hour-level rebalancing.** Add an hour-of-day analysis (not shown in this scoped EDA) to move bikes to morning/evening commute corridors and entertainment zones on weekends.6. **Invest in all-weather readiness.** Since heavy rain/snow collapses demand, explore covered parking, weather insurance messaging or partnership with public transit during extreme weather to protect brand trust.## Limitations- Revenue proxy assumes a fixed price per ride; actual pricing, promotions, subscription models and margins are not in the data.- The dataset covers only two years and one city context; demand patterns may differ in new markets or with new competitors.- Causal claims are avoided; the analysis shows association, not causation.